In [ ]:
# !pip install graphviz

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_graphviz
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import graphviz

In [ ]:
# Define the Raw Data URLs
batter_url = "https://github.com/willcreager/willcreager.github.io/raw/refs/heads/main/docs/data/csvs/MLBData-ABSChallengeablePitches-Batters-FullSeason.csv"
defense_url = "https://github.com/willcreager/willcreager.github.io/raw/refs/heads/main/docs/data/csvs/MLBData-ABSChallengeablePitches-Defense-FullSeason.csv"

In [ ]:
# Create function to execute clustering pipeline
def run_decision_tree_pipeline(X, y, position_label, use_balancing=False):
    print(f"\n==================================================================")
    print(f"🌲 RUNNING COMPREHENSIVE DECISION TREE PIPELINE FOR: {position_label.upper()}")
    if use_balancing:
        print(f"CLASS BALANCE CORRECTION: ACTIVE")
    print(f"==================================================================")
    print(f"Total available rows: {len(X)} | Target classes: {np.unique(y)}")

    # Create disjoint Training and Testing sets (80/20 split)
    # Stratify handles class imbalance during splits to prevent data starvation
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
    print(f"Disjoint Splits Created: Training Set = {X_train.shape[0]} rows | Testing Set = {X_test.shape[0]} rows")

    # Dynamically inject class weights based on use_balancing choice
    weight_param = "balanced" if use_balancing else None

    # Create 3 distinct tree variations
    models_config = {
        "Tree 1: Shallow Gini Split (Max Depth 3)": DecisionTreeClassifier(
            criterion="gini", max_depth=3, class_weight=weight_param, random_state=42
        ),
        "Tree 2: Deep Entropy Split (Max Depth 5)": DecisionTreeClassifier(
            criterion="entropy", max_depth=5, class_weight=weight_param, random_state=42
        ),
        "Tree 3: Regularized Gini Split (Max Depth 4, Min Samples Leaf 20)": DecisionTreeClassifier(
            criterion="gini", max_depth=4, min_samples_leaf=20, class_weight=weight_param, random_state=42
        )
    }

    # Store test predictions to generate downstream evaluations
    fitted_trees = {}

    # Loop through and execute evaluations across all 3 specifications
    for name, clf in models_config.items():
        print(f"\n⚡ Training Configuration: {name}...")
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        fitted_trees[name] = (clf, y_pred)

        # Output evaluation metrics text blocks
        acc = accuracy_score(y_test, y_pred)
        print(f"✅ Holdout Test Accuracy Score: {acc:.4f}")
        print(f"Confusion Matrix:")
        print(confusion_matrix(y_test, y_pred))
        print(classification_report(y_test, y_pred))

    # PLOT 1: RECONSTRUCTED SIDE-BY-SIDE CONFUSION HEATMAPS
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for i, (name, (clf, y_pred)) in enumerate(fitted_trees.items()):
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", ax=axes[i],
                    xticklabels=clf.classes_, yticklabels=clf.classes_)
        axes[i].set_title(f"{name}\nAcc: {accuracy_score(y_test, y_pred):.3f}", fontweight="bold")
        axes[i].set_xlabel("Predicted Label")
        axes[i].set_ylabel("Actual Label")
    plt.suptitle(f"{position_label}: Confusion Matrix Layout Profiles", fontsize=14, fontweight="bold", y=1.05)
    plt.tight_layout()
    plt.show()

    # PLOT 2: STRUCTURAL GEOMETRY RENDERS
    for name, (clf, y_pred) in fitted_trees.items():
        plt.figure(figsize=(20, 8))
        plot_tree(clf,
                  feature_names=X.columns.tolist(),
                  class_names=[str(c) for c in clf.classes_],
                  filled=True,
                  rounded=True,
                  fontsize=10)
        plt.title(f"Structural Split Map: {name} ({position_label})", fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.show()

    # GRAPHVIZ GRAPH STRUCTURE EXPORT FOR WEB INSERTION
    # Generates clear, high-resolution vector images
    print("\n--- GRAPHVIZ WEBPAGE HIGH-RES ASSET EXPORT CHECKS ---")
    clean_label = position_label.lower().replace(" ", "_").replace("/", "_")

    for idx, (name, (clf, y_pred)) in enumerate(fitted_trees.items()):
        dot_data = export_graphviz(
            clf,
            out_file=None,
            feature_names=X.columns.tolist(),
            class_names=[str(c) for c in clf.classes_],
            filled=True,
            rounded=True,
            special_characters=True
        )
        graph = graphviz.Source(dot_data)

        # Save production assets locally
        filename = f"tree_v{idx+1}_{clean_label}"
        graph.render(filename, format="png")
        print(f"📁 Successfully compiled and saved web-ready vector: {filename}.png")

    print("\n==================================================================")
    print(f"🏁 COMPREHENSIVE DECISION TREE PIPELINE COMPLETE FOR: {position_label.upper()}")
    print(f"==================================================================\n")


# Using Decision Trees to predict whether batter will challenge

In [ ]:
# Load batting data
df_batters = pd.read_csv(batter_url, encoding="latin1")

# Remove rows with null pitch locations
df_batters = df_batters.dropna(subset=["plateTimeY", "plateTimeX"])

# Feature Engineering: Calculate vertical distance deviations from zone edges
# Positive values mean above the edge, negative values mean below the edge
df_batters["plateTimeY_topDiff"] = (
    df_batters["plateTimeY"] - df_batters["szTop"]
)
df_batters["plateTimeY_BotDiff"] = (
    df_batters["plateTimeY"] - df_batters["szBottom"]
)

# Set challengesLeft based on inningHalf (0 = Away, 1 = Home)
df_batters["challengesLeft"] = np.where(
    df_batters["inningHalf"] == 0,
    df_batters["awayChallengesLeft"],
    df_batters["homeChallengesLeft"],
)

# Set netScore based on inningHalf (0 = Away, 1 = Home)
# Positive if batting team is winning, 0 if tied, negative if losing
df_batters["netScore"] = np.where(
    df_batters["inningHalf"] == 0,
    df_batters["prePlayAwayScore"]-df_batters["prePlayHomeScore"],
    df_batters["prePlayHomeScore"]-df_batters["prePlayAwayScore"],
)

# Add pitchTypeGroup feature (1=fastballs, 2=breaking balls, 3=other)
# Define the conditions based on pitchTypeId
conditions = [
    df_batters["pitchTypeId"].isin([5, 6, 7, 12]),  # Fastballs
    df_batters["pitchTypeId"].isin([2, 3, 10, 13, 14, 15]),  # Breaking Balls
]

# Define the corresponding values for each condition
choices = [1, 2]

# Apply the conditions; default to 3 (Other) if no condition is met
df_batters["pitchTypeGroup"] = np.select(conditions, choices, default=3)

# Create list of columns to use as features
feature_cols = [
    "inning",
    "prePitchBalls",
    "prePitchStrikes",
    "runnerOn1st",
    "runnerOn2nd",
    "runnerOn3rd",
    "outs",
    "plateTimeX",
    "plateTimeY_topDiff",
    "plateTimeY_BotDiff",
    "startSpeed",
    "challengesLeft",
    "netScore",
    "pitchTypeGroup",
]

# Create the features DataFrame
X = df_batters[feature_cols]
# Create classes
y = df_batters["hasABSChallenge"].map({1: "Challenged", 0: "Not Challenged"})

print(f"There are {len(X)} rows in the cleaned Features dataset.")

# View the features DataFrame
X.head(20)

In [ ]:
run_decision_tree_pipeline(X, y, "Batter Challenge", use_balancing=True)

# Using Decision Trees to predict whether a batter's challenge will be overturned

In [ ]:
# Removed pitch location data from feature list since those values
#  can explicitly determine if a pitch was a ball or strike
feature_cols2 = [
    "inning",
    "prePitchBalls",
    "prePitchStrikes",
    "runnerOn1st",
    "runnerOn2nd",
    "runnerOn3rd",
    "outs",
    "startSpeed",
    "challengesLeft",
    "netScore",
    "pitchTypeGroup",
]

# Create new dataset that only contains challenged pitches
df_batters_ChallengedPitches = df_batters[df_batters["hasABSChallenge"] == 1]

# Create the features DataFrame
X = df_batters_ChallengedPitches[feature_cols2]
# Create classes
y = df_batters_ChallengedPitches["isOverturned"].map({1: "Overturned", 0: "Confirmed"})

print(f"There are {len(X)} rows in the cleaned Features dataset.")

# View the features DataFrame
X.head(20)

In [ ]:
run_decision_tree_pipeline(X, y, "Batter Challenge Outcome")

# Using Decision Trees to predict whether catcher/pitcher will challenge

In [ ]:
# Load defensive data
df_def = pd.read_csv(defense_url, encoding="latin1")

# Remove rows with null pitch locations
df_def = df_def.dropna(subset=["plateTimeY", "plateTimeX"])

# Feature Engineering: Calculate vertical distance deviations from zone edges
# Positive values mean above the edge, negative values mean below the edge
df_def["plateTimeY_topDiff"] = (
    df_def["plateTimeY"] - df_def["szTop"]
)
df_def["plateTimeY_BotDiff"] = (
    df_def["plateTimeY"] - df_def["szBottom"]
)

# Set challengesLeft based on inningHalf
df_def["challengesLeft"] = np.where(
    df_def["inningHalf"] == 1,
    df_def["awayChallengesLeft"],
    df_def["homeChallengesLeft"],
)

# Set netScore based on inningHalf
# Positive if defensive team is winning, 0 if tied, negative if losing
df_def["netScore"] = np.where(
    df_def["inningHalf"] == 1,
    df_def["prePlayAwayScore"]-df_def["prePlayHomeScore"],
    df_def["prePlayHomeScore"]-df_def["prePlayAwayScore"],
)

# Add pitchTypeGroup feature (1=fastballs, 2=breaking balls, 3=other)
# Define the conditions based on pitchTypeId
conditions = [
    df_def["pitchTypeId"].isin([5, 6, 7, 12]),  # Fastballs
    df_def["pitchTypeId"].isin([2, 3, 10, 13, 14, 15]),  # Breaking Balls
]

# Define the corresponding values for each condition
choices = [1, 2]

# Apply the conditions; default to 3 (Other) if no condition is met
df_def["pitchTypeGroup"] = np.select(conditions, choices, default=3)

# Create list of columns to use as features
feature_cols = [
    "inning",
    "prePitchBalls",
    "prePitchStrikes",
    "runnerOn1st",
    "runnerOn2nd",
    "runnerOn3rd",
    "outs",
    "plateTimeX",
    "plateTimeY_topDiff",
    "plateTimeY_BotDiff",
    "startSpeed",
    "challengesLeft",
    "netScore",
    "pitchTypeGroup",
]

# Create the features DataFrame
X = df_def[feature_cols]
# Create classes
y = df_def["hasABSChallenge"].map({1: "Challenged", 0: "Not Challenged"})

print(f"There are {len(X)} rows in the cleaned Features dataset.")

# View the features DataFrame
X.head(20)

In [ ]:
run_decision_tree_pipeline(X, y, "Defense Challenge", use_balancing=True)

# Using NB to predict whether a catcher/pitcher's challenge will be overturned

In [ ]:
# Removed pitch location data from feature list since those values
#  can explicitly determine if a pitch was a ball or strike
feature_cols2 = [
    "inning",
    "prePitchBalls",
    "prePitchStrikes",
    "runnerOn1st",
    "runnerOn2nd",
    "runnerOn3rd",
    "outs",
    "startSpeed",
    "challengesLeft",
    "netScore",
    "pitchTypeGroup",
]

# Create new dataset that only contains challenged pitches
df_def_ChallengedPitches = df_def[df_def["hasABSChallenge"] == 1]

# Create the features DataFrame
X = df_def_ChallengedPitches[feature_cols2]
# Create classes
y = df_def_ChallengedPitches["isOverturned"].map({1: "Overturned", 0: "Confirmed"})

print(f"There are {len(X)} rows in the cleaned Features dataset.")

# View the features DataFrame
X.head(20)

In [ ]:
run_decision_tree_pipeline(X, y, "Defense Challenge Outcome")

# Create diagrams for website

In [ ]:
# Initialize side-by-side plot layout
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ----------------------------------------------------
# LEFT PLOT: Spatial Feature Partitioning Space
# ----------------------------------------------------
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 10)

# Create color-coded regional splits representing pure child subsets
ax1.add_patch(patches.Rectangle((0, 0), 5, 10, facecolor="#d4edda", alpha=0.6, label="Region A (Confirmed)"))
ax1.add_patch(patches.Rectangle((5, 0), 5, 4, facecolor="#f8d7da", alpha=0.6, label="Region B (Overturned)"))
ax1.add_patch(patches.Rectangle((5, 4), 5, 6, facecolor="#fff3cd", alpha=0.6, label="Region C (Overturned)"))

# Draw the explicit boundary lines executed by the tree nodes
ax1.axvline(x=5, color="black", linestyle="-", linewidth=2.5)
ax1.text(4.6, 5, r"Split 1: Feature X $\leq$ 5", rotation=90, fontsize=11, fontweight="bold")

ax1.axhline(y=4, xmin=0.5, xmax=1.0, color="darkred", linestyle="-", linewidth=2.5)
ax1.text(6, 4.3, r"Split 2: Feature Y $\leq$ 4", color="darkred", fontsize=11, fontweight="bold")

# Scatter mock data points inside partitioned spaces
ax1.scatter([2, 3, 1, 4, 2.5], [5, 6, 7, 4.5, 8], color="green", s=70, edgecolor="black", zorder=5)
ax1.scatter([6, 7.5, 8, 9.5], [2, 1, 3, 1.5], color="red", s=70, edgecolor="black", zorder=5)
ax1.scatter([6.5, 8.5, 7, 9], [6, 8, 7.5, 9], color="darkorange", s=70, edgecolor="black", zorder=5)

ax1.set_title("1A: Recursive Feature Space Partitioning", fontsize=12, fontweight="bold", pad=15)
ax1.set_xlabel("Feature X", fontsize=11)
ax1.set_ylabel("Feature Y", fontsize=11)
ax1.grid(True, linestyle=":", alpha=0.6)

# ----------------------------------------------------
# RIGHT PLOT: Logical Flowchart System Layout
# ----------------------------------------------------
# CRITICAL FIX: Lock the coordinate dimensions before turning the background grid lines off
ax2.set_xlim(0, 12)
ax2.set_ylim(0, 10)
ax2.axis("off")

# Render text boxes representing tree nodes cleanly
bbox_props = dict(boxstyle="round,pad=0.5", fc="#e2ebfa", ec="black", lw=1.5)
leaf_props_a = dict(boxstyle="round,pad=0.5", fc="#d4edda", ec="green", lw=1.5)
leaf_props_b = dict(boxstyle="round,pad=0.5", fc="#f8d7da", ec="red", lw=1.5)
leaf_props_c = dict(boxstyle="round,pad=0.5", fc="#fff3cd", ec="darkorange", lw=1.5)

# Render Parent, Interior, and Leaf nodes matching Left Spatial coordinates seamlessly
ax2.text(6, 9, "Root Node\nFeature X $\leq$ 5", ha="center", va="center", size=11, bbox=bbox_props, fontweight="bold")
ax2.text(2, 5, "Leaf Region A\nPredict: Class A", ha="center", va="center", size=10, bbox=leaf_props_a)
ax2.text(10, 5, "Interior Node\nFeature Y $\leq$ 4", ha="center", va="center", size=11, bbox=bbox_props, fontweight="bold")
ax2.text(8, 1.5, "Leaf Region B\nPredict: Class B", ha="center", va="center", size=10, bbox=leaf_props_b)
ax2.text(12, 1.5, "Leaf Region C\nPredict: Class C", ha="center", va="center", size=10, bbox=leaf_props_c)

# Draw arrows linking branching structures using precise coordinate points
arrow_style = dict(arrowstyle="->", lw=2, color="black")
ax2.annotate("True", xy=(2, 5.8), xytext=(5, 8.2), arrowprops=arrow_style, ha="center")
ax2.annotate("False", xy=(10, 5.8), xytext=(7, 8.2), arrowprops=arrow_style, ha="center")
ax2.annotate("True", xy=(8, 2.3), xytext=(9.5, 4.2), arrowprops=arrow_style, ha="center")
ax2.annotate("False", xy=(12, 2.3), xytext=(10.5, 4.2), arrowprops=arrow_style, ha="center")

ax2.set_title("1B: Corresponding Binary Decision Tree Flow", fontsize=12, fontweight="bold", pad=15)

# Adjust margins to ensure titles and labels display fully
plt.subplots_adjust(left=0.08, right=0.92, bottom=0.12, top=0.85, wspace=0.25)
plt.suptitle("Figure 1: Conceptual Core Mechanics of a Decision Tree Model", fontsize=14, fontweight="bold", y=0.96)

plt.show()

In [ ]:
# Generate probability spectrum
p = np.linspace(0.001, 0.999, 500)

# Compute impurity metrics mathematically over range values
entropy = -p * np.log2(p) - (1 - p) * np.log2(1 - p)
gini = 2 * p * (1 - p)
scaled_entropy = 0.5 * entropy

plt.figure(figsize=(9, 5.5))

# Plot mathematical impurity functional representations
plt.plot(p, entropy, label="Entropy (Max Disordered Value = 1.0)", color="darkblue", linewidth=2.5)
plt.plot(p, gini, label="Gini Impurity (Max Disordered Value = 0.5)", color="crimson", linewidth=2.5, linestyle="--")
plt.plot(p, scaled_entropy, label="Scaled Entropy ($0.5 \\times$ Entropy)", color="dimgray", linewidth=1.5, linestyle=":")

# Draw horizontal/vertical threshold anchors mapping the optimal peaks
plt.axvline(x=0.5, color="black", linestyle=":", alpha=0.7, lw=1.5)
plt.axhline(y=0.5, color="crimson", linestyle=":", alpha=0.5, lw=1.2)
plt.axhline(y=1.0, color="darkblue", linestyle=":", alpha=0.5, lw=1.2)

# Scatter highlighting labels at maximum class split chaos
plt.scatter([0.5, 0.5], [0.5, 1.0], color="black", s=50, zorder=5)
plt.text(0.501, 1.02, "Max Chaos Point, Entropy", fontsize=10, fontweight="bold")
plt.text(0.501, 0.52, "Max Chaos Point, Gini Impurity", fontsize=10, fontweight="bold")

plt.title("Figure 2: Mathematical Impurity Curves for Gini and Entropy Metrics", fontsize=13, fontweight="bold", pad=15)
plt.xlabel("Probability Value of Class $p$ (e.g., Overturned Challenge Proportion)", fontsize=11)
plt.ylabel("Calculated Statistical Impurity Output", fontsize=11)
plt.xlim(0, 1)
plt.ylim(0, 1.1)
plt.legend(loc="upper left", frameon=True, shadow=True, fontsize=10)
plt.grid(True, linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()